### Initialization

Run the following setup cells from the Decoders working directory in the qec environment. They load this notebook's configuration, paths and dependencies; subsequent experiments reuse these definitions and should be read in execution order.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
EXPERIMENT_DIR = ROOT / 'experiment/research/bp_failure_dynamics/stages/03_logical_degeneracy'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

_data_paths = ExperimentPaths(ROOT, EXPERIMENT_DIR / 'config/paths.json')


# BP instability and logical degeneracy

This notebook independently reconstructs the distance-5 weight-2/3 decoder-input population, computes exact logical-sector posterior masses on matched syndrome-valid cases, and measures logical susceptibility under prior-LLR perturbations. Sampled truth, maximum-mass sector, syndrome validity, and BP logical correctness remain distinct quantities.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from collections import defaultdict
import csv
import json
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.special import logsumexp
from scipy.stats import rankdata, spearmanr, wilcoxon
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from beliefmatching import detector_error_model_to_check_matrices
from codes.ca_code import CA_9_2_5
from ldpc import BpDecoder
from noises_ca import CircuitBuilder, ErrorRate

CONFIG = json.loads((EXPERIMENT_DIR / 'config/instability_degeneracy_experiments.json').read_text(encoding='utf-8'))
# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.
# Expected relationship: long_iterations >= benchmark_iterations >= diagnostic_iterations >= 3.
# ms_scaling_factor: finite and positive.
# late_fraction: in (0, 1].
# message_zero_tolerance: non-negative.
# primary_stability_threshold: chosen from stability_thresholds.
# stability_thresholds: each item is finite and positive.
# stable_matched_pairs: positive.
# unstable_matched_pairs: positive.
# exact_enumeration_chunk_size: positive.
# exact_max_nullity: non-negative.
# susceptibility_k_values: each item is positive.
# perturbation_eta_values: each item is finite and positive.
# Expected relationship: num_perturbations >= random_subset_replicates > 1.
# probability_clip: in (0, 0.5).
# cv_folds: at least 2.
# bootstrap_repetitions: positive.
DATA = _data_paths.run()

from experiment.utils.bp_trace import check_bp_cache
check_bp_cache(_data_paths, CONFIG)

INPUT = _data_paths.input('error_weight')
STATS = _data_paths.stats()
TRACE = _data_paths.trace()
STATS.mkdir(parents=True, exist_ok=True)
TRACE.mkdir(parents=True, exist_ok=True)

from experiment.utils import misc

def log(level, payload):
    return misc.log(level, payload, CONFIG=CONFIG)

sns.set_theme(style='whitegrid')
log(2, {'root': str(ROOT), 'stats': str(STATS), 'trace': str(TRACE)})


In [ ]:
DTYPE = {
    'p': float,
    'err': str,
    'det': str,
    'pred_b': str,
    'pred_t': str,
    'true': str,
}

weight2 = pd.read_csv(_data_paths.input('weight_2_data_csv'), dtype=DTYPE)
weight3 = pd.read_csv(_data_paths.input('weight_3_data_csv'), dtype=DTYPE)

required = set(DTYPE)
for frame, weight in ((weight2, 2), (weight3, 3)):
    assert required <= set(frame.columns) and frame.notna().all().all()
    assert frame.err.str.count('1').eq(weight).all()
    frame['syndrome_weight'] = frame.det.str.count('1').astype(int)

input_audit = pd.DataFrame([{
    'weight2_rows': len(weight2),
    'weight3_rows': len(weight3),
    'weight2_unique_decoder_inputs': len(weight2.drop_duplicates(['p', 'det', 'true'])),
    'weight3_unique_decoder_inputs': len(weight3.drop_duplicates(['p', 'det', 'true'])),
}])

input_audit.to_csv(_data_paths.output('instability_degeneracy_input_audit_csv'), index=False)
input_audit


## Instrumented parallel minimum-sum BP

Prepare the native BP trajectory and ambiguity-analysis helpers. Message residual describes dynamical motion; the later logical-sector calculation measures a different property of the posterior.


In [ ]:
from experiment.utils import bp_message_dynamics

Context = bp_message_dynamics.Context
MessageTrace = bp_message_dynamics.MessageTrace
CACHE = {}

def context_for(p):
    return bp_message_dynamics.context_for(
        p,
        CACHE=CACHE,
        CA_9_2_5=CA_9_2_5,
        CircuitBuilder=CircuitBuilder,
        Context=Context,
        ErrorRate=ErrorRate,
        detector_error_model_to_check_matrices=detector_error_model_to_check_matrices,
    )

from experiment.utils import misc

bit_array = misc.bit_array
bit_text = misc.bit_text
logical = bp_message_dynamics.logical

def trace_min_sum(
    context,
    syndrome,
    iterations,
    stop_early=False,
):
    return bp_message_dynamics.trace_min_sum(context, syndrome, iterations, stop_early, CONFIG=CONFIG, MessageTrace=MessageTrace)

def trace_prefix(trace, iterations):
    return bp_message_dynamics.trace_prefix(trace, iterations, MessageTrace=MessageTrace)

def benchmark_state(trace):
    return bp_message_dynamics.benchmark_state(trace, CONFIG=CONFIG)

def edge_metrics(context, trace):
    return bp_message_dynamics.edge_metrics(context, trace, CONFIG=CONFIG)

def variable_metrics(context, trace, edges):
    return bp_message_dynamics.variable_metrics(context, trace, edges, CONFIG=CONFIG)

classify_outcome = bp_message_dynamics.classify_outcome
METRICS = ['max_normalized_late_residual', 'mean_normalized_late_residual', 'max_flip_rate', 'mean_flip_rate', 'max_period2_strength', 'mean_message_instability', 'mean_belief_instability', 'mean_extrinsic_conflict']


In [ ]:
SECTORS = ['00', '01', '10', '11']

from experiment.utils import bp_message_dynamics

def normalized_late_residual(context, trace, endpoint):
    return bp_message_dynamics.normalized_late_residual(context, trace, endpoint, CONFIG=CONFIG)

def logical_label(context, hard):
    return bit_text(logical(context, hard))

def variable_instability_score(context, trace):
    messages = trace.c2v[1:CONFIG['diagnostic_iterations'] + 1]
    delta = np.abs(np.diff(messages, axis=0))
    late_n = max(1, int(np.ceil(CONFIG['late_fraction'] * len(delta))))
    scale = np.abs(context.prior_llr[context.edge_variables]) + np.finfo(float).eps
    edge_score = np.mean(delta[-late_n:] / scale, axis=0)
    score = np.zeros(context.H.shape[1], dtype=float)

    for variable, edge_ids in enumerate(context.variable_edges):
        assert len(edge_ids) > 0
        score[variable] = edge_score[edge_ids].mean()

    return score

def outcome_class(stable, logical_correct):
    prefix = 'stable' if stable else 'unstable'
    suffix = 'correct' if logical_correct else 'logical-wrong'
    return f'{prefix}-valid-{suffix}'


## Population reconstruction and outcome classification

Reconstruct the fixed-weight inputs and label their syndrome/logical outcomes. These labels define the diagnostic population from which the matched comparisons are formed.


In [ ]:
frames = []
for source, error_weight in ((weight2, 2), (weight3, 3)):
    frame = source.drop_duplicates(['p', 'det', 'true']).copy()
    frame['error_weight'] = error_weight
    frames.append(frame)

population = pd.concat(frames, ignore_index=True).sample(frac=1, random_state=CONFIG['random_seed']).reset_index(drop=True)
population_rows = []
for sample_id, row in enumerate(tqdm(population.itertuples(index=False), total=len(population), desc='Classify population', disable=CONFIG['print_level'] == 1)):
    context = context_for(row.p)
    syndrome = bit_array(row.det)
    truth = bit_array(row.true)
    error = bit_array(row.err)
    trace = trace_min_sum(context, syndrome, CONFIG['long_iterations'])
    first = trace.first_valid_iteration
    benchmark_iteration = first if first is not None and first <= CONFIG['benchmark_iterations'] else CONFIG['benchmark_iterations']
    hard = trace.hard[benchmark_iteration]
    syndrome_consistent = bool(trace.syndrome_consistent[benchmark_iteration])
    predicted_sector = logical_label(context, hard)
    logical_correct = predicted_sector == row.true
    late_residual = normalized_late_residual(context, trace, CONFIG['diagnostic_iterations'])
    stable = late_residual < CONFIG['primary_stability_threshold']
    residual_weight = np.nan
    classification = 'syndrome-invalid'

    if syndrome_consistent:
        residual = error ^ hard
        assert not np.any(context.H @ residual % 2)
        residual_weight = int(residual.sum())
        classification = outcome_class(stable, logical_correct)

    population_rows.append({
        'sample_id': sample_id,
        'p': row.p,
        'error_weight': row.error_weight,
        'syndrome_weight': row.syndrome_weight,
        'error_support': row.err,
        'syndrome': row.det,
        'true_sector': row.true,
        'benchmark_iteration': benchmark_iteration,
        'benchmark_syndrome_consistent': syndrome_consistent,
        'benchmark_sector': predicted_sector,
        'benchmark_logical_correct': logical_correct,
        'max_normalized_late_residual': late_residual,
        'stable': stable,
        'residual_weight': residual_weight,
        'outcome_class': classification,
    })

degeneracy_population = pd.DataFrame(population_rows)
assert len(degeneracy_population) == 23178 and degeneracy_population.sample_id.is_unique
degeneracy_population.to_csv(_data_paths.output('instability_degeneracy_population_cases_csv'), index=False)
population_summary = degeneracy_population.groupby(['p', 'error_weight', 'outcome_class'], as_index=False).size().rename(columns={'size': 'cases'})

population_summary.to_csv(_data_paths.output('instability_degeneracy_population_summary_csv'), index=False)
sensitivity_rows = []
for threshold in CONFIG['stability_thresholds']:
    classified = degeneracy_population.copy()
    classified['threshold'] = threshold
    classified['threshold_stable'] = classified.max_normalized_late_residual < threshold
    classified['threshold_outcome_class'] = 'syndrome-invalid'
    valid = classified.benchmark_syndrome_consistent
    classified.loc[valid, 'threshold_outcome_class'] = np.where(classified.loc[valid, 'threshold_stable'], 'stable-valid-', 'unstable-valid-') + np.where(classified.loc[valid, 'benchmark_logical_correct'], 'correct', 'logical-wrong')
    sensitivity_rows.append(classified.groupby(['threshold', 'p', 'error_weight', 'threshold_outcome_class'], as_index=False).size().rename(columns={'size': 'cases'}))

stability_sensitivity = pd.concat(sensitivity_rows, ignore_index=True)
assert stability_sensitivity.groupby('threshold').cases.sum().eq(len(degeneracy_population)).all()
stability_sensitivity.to_csv(_data_paths.output('instability_degeneracy_stability_sensitivity_csv'), index=False)
population_summary


## Matched logical-mass cohorts

Select comparable stable-wrong and stable-correct cases before evaluating exact logical masses. Matching controls the configured covariates but does not turn this case-control sample into a physical-noise LER estimate.


In [ ]:
def matched_pair_sample(
    frame,
    left_class,
    right_class,
    keys,
    limit,
    family,
    seed,
):
    left = frame[frame.outcome_class == left_class].sample(frac=1, random_state=seed)
    right = frame[frame.outcome_class == right_class].sample(frac=1, random_state=seed + 1)
    pairs = []
    pair_index = 0

    for values, left_group in left.groupby(keys, sort=True):
        values = values if isinstance(values, tuple) else (values,)
        mask = np.ones(len(right), dtype=bool)
        for key, value in zip(keys, values):
            mask &= right[key].to_numpy() == value
        right_group = right.loc[mask]
        count = min(len(left_group), len(right_group))

        for left_row, right_row in zip(left_group.head(count).itertuples(index=False), right_group.head(count).itertuples(index=False)):
            pair_id = f'{family}-{pair_index}'
            pairs.extend([{'matched_pair_id': pair_id, 'match_family': family, **left_row._asdict()}, {'matched_pair_id': pair_id, 'match_family': family, **right_row._asdict()}])
            pair_index += 1

    matched = pd.DataFrame(pairs)
    available_ids = pd.Series(matched.matched_pair_id.unique())
    assert len(available_ids) >= limit, (family, len(available_ids), limit)
    selected_ids = available_ids.sample(limit, random_state=seed + 2).to_numpy()
    result = matched[matched.matched_pair_id.isin(selected_ids)].copy()
    assert result.groupby('matched_pair_id').size().eq(2).all()
    assert result.groupby('matched_pair_id').outcome_class.nunique().eq(2).all()

    return result

stable_matched = matched_pair_sample(degeneracy_population, 'stable-valid-correct', 'stable-valid-logical-wrong', ['p', 'error_weight', 'syndrome_weight', 'residual_weight'], CONFIG['stable_matched_pairs'], 'stable', CONFIG['random_seed'])
unstable_matched = matched_pair_sample(degeneracy_population, 'unstable-valid-correct', 'unstable-valid-logical-wrong', ['p', 'error_weight', 'syndrome_weight'], CONFIG['unstable_matched_pairs'], 'unstable', CONFIG['random_seed'] + 100)
matched_cases = pd.concat([stable_matched, unstable_matched], ignore_index=True)
assert matched_cases.sample_id.is_unique
matching_audit = matched_cases.groupby(['match_family', 'outcome_class'], as_index=False).agg(cases=('sample_id', 'size'), matched_pairs=('matched_pair_id', 'nunique'))

matching_audit.to_csv(_data_paths.output('logical_ambiguity_selection_audit_csv'), index=False)
matching_audit


## Experiment 1: exact logical-sector masses

Enumerate the syndrome coset and sum prior probability within each logical sector. The normalization audit checks this finite-code calculation before its gap and entropy are used as ambiguity measurements.


In [ ]:
def gf2_rref(matrix, rhs=None):
    matrix = np.asarray(matrix, dtype=np.uint8).copy() % 2
    augmented = matrix if rhs is None else np.column_stack([matrix, np.asarray(rhs, dtype=np.uint8) % 2])
    row = 0
    pivots = []

    for column in range(matrix.shape[1]):
        candidates = np.flatnonzero(augmented[row:, column])
        if not len(candidates):
            continue
        pivot = row + int(candidates[0])
        augmented[[row, pivot]] = augmented[[pivot, row]]
        other = np.flatnonzero(augmented[:, column])
        other = other[other != row]
        augmented[other] ^= augmented[row]
        pivots.append(column)
        row += 1

        if row == matrix.shape[0]:
            break

    return (augmented, pivots)

def gf2_solution_and_nullspace(H, syndrome):
    reduced, pivots = gf2_rref(H, syndrome)
    rank = len(pivots)
    assert not np.any((reduced[rank:, :-1].sum(axis=1) == 0) & (reduced[rank:, -1] == 1))
    free = [column for column in range(H.shape[1]) if column not in pivots]
    particular = np.zeros(H.shape[1], dtype=np.uint8)

    for row, pivot in enumerate(pivots):
        particular[pivot] = reduced[row, -1]
    basis = np.zeros((len(free), H.shape[1]), dtype=np.uint8)

    for basis_row, free_column in enumerate(free):
        basis[basis_row, free_column] = 1
        for row, pivot in enumerate(pivots):
            basis[basis_row, pivot] = reduced[row, free_column]

    assert np.array_equal(H @ particular % 2, np.asarray(syndrome, dtype=np.uint8))
    assert not np.any(H @ basis.T % 2)

    return (particular, basis, rank)

def enumerate_kernel_states(basis):
    dimension = basis.shape[0]
    assert dimension <= CONFIG['exact_max_nullity']
    count = 1 << dimension
    states = np.empty((count, basis.shape[1]), dtype=np.uint8)
    chunk = CONFIG['exact_enumeration_chunk_size']
    shifts = np.arange(dimension, dtype=np.uint64)

    for start in tqdm(range(0, count, chunk), desc='Enumerate kernel', disable=CONFIG['print_level'] == 1):
        stop = min(count, start + chunk)
        coordinates = (np.arange(start, stop, dtype=np.uint64)[:, None] >> shifts & 1).astype(np.uint8)
        states[start:stop] = coordinates @ basis % 2

    return states

REFERENCE_CONTEXT = context_for(float(population.p.iloc[0]))
ZERO_SYNDROME = np.zeros(REFERENCE_CONTEXT.H.shape[0], dtype=np.uint8)
_, NULLSPACE_BASIS, H_RANK = gf2_solution_and_nullspace(REFERENCE_CONTEXT.H, ZERO_SYNDROME)
KERNEL_STATES = enumerate_kernel_states(NULLSPACE_BASIS)
KERNEL_LOGICAL = KERNEL_STATES @ REFERENCE_CONTEXT.L.T % 2
KERNEL_SECTOR = np.asarray([int(bit_text(label), 2) for label in KERNEL_LOGICAL], dtype=np.uint8)
assert len(KERNEL_STATES) == 1 << NULLSPACE_BASIS.shape[0]

def compute_exact_logical_masses(
    H,
    L,
    syndrome,
    priors,
):
    assert np.array_equal(H, REFERENCE_CONTEXT.H) and np.array_equal(L, REFERENCE_CONTEXT.L)
    particular, basis, rank = gf2_solution_and_nullspace(H, syndrome)
    assert np.array_equal(basis, NULLSPACE_BASIS)
    base = float(np.log1p(-priors).sum())
    log_odds = np.log(priors) - np.log1p(-priors)
    particular_sector = int(bit_text(L @ particular % 2), 2)
    log_Z = np.full(4, -np.inf, dtype=float)
    chunk = CONFIG['exact_enumeration_chunk_size']

    for start in range(0, len(KERNEL_STATES), chunk):
        stop = min(len(KERNEL_STATES), start + chunk)
        errors = KERNEL_STATES[start:stop] ^ particular
        log_probability = base + errors @ log_odds
        sectors = KERNEL_SECTOR[start:stop] ^ particular_sector

        for sector in range(4):
            selected = log_probability[sectors == sector]
            if len(selected):
                log_Z[sector] = np.logaddexp(log_Z[sector], logsumexp(selected))

    normalization = logsumexp(log_Z)
    probabilities = np.exp(log_Z - normalization)
    assert np.isclose(probabilities.sum(), 1.0, atol=1e-12)

    return (log_Z, probabilities, rank, basis.shape[0])

def logical_ambiguity_metrics(log_Z, true_sector):
    order = np.argsort(log_Z)[::-1]
    probabilities = np.exp(log_Z - logsumexp(log_Z))
    entropy = float(-np.sum(probabilities * np.log(probabilities)))
    alternatives = np.delete(log_Z, true_sector)

    return {
        'top_two_log_mass_gap': float(log_Z[order[0]] - log_Z[order[1]]),
        'logical_entropy': entropy,
        'normalized_logical_entropy': entropy / np.log(4),
        'true_sector_gap': float(log_Z[true_sector] - np.max(alternatives)),
        'maximum_mass_sector': SECTORS[int(order[0])],
        'true_sector_is_maximum_mass': int(order[0] == true_sector),
    }

def brute_force_logical_masses(
    H,
    L,
    syndrome,
    priors,
):
    rows = []
    for value in range(1 << H.shape[1]):
        error = (value >> np.arange(H.shape[1]) & 1).astype(np.uint8)
        if np.array_equal(H @ error % 2, syndrome):
            sector = int(bit_text(L @ error % 2), 2)
            log_probability = np.where(error, np.log(priors), np.log1p(-priors)).sum()
            rows.append((sector, log_probability))

    return np.asarray([logsumexp([score for label, score in rows if label == sector]) for sector in range(1 << L.shape[0])])

toy_H = np.asarray([[1, 1, 0, 1], [0, 1, 1, 0]], dtype=np.uint8)
toy_L = np.asarray([[1, 0, 1, 0]], dtype=np.uint8)
toy_syndrome = np.asarray([1, 0], dtype=np.uint8)
toy_priors = np.asarray([0.08, 0.13, 0.21, 0.17])
toy_particular, toy_basis, _ = gf2_solution_and_nullspace(toy_H, toy_syndrome)
toy_states = enumerate_kernel_states(toy_basis)
toy_log_Z = np.full(2, -np.inf)

for error in toy_states ^ toy_particular:
    sector = int((toy_L @ error)[0] % 2)
    score = np.where(error, np.log(toy_priors), np.log1p(-toy_priors)).sum()
    toy_log_Z[sector] = np.logaddexp(toy_log_Z[sector], score)

np.testing.assert_allclose(toy_log_Z, brute_force_logical_masses(toy_H, toy_L, toy_syndrome, toy_priors), atol=1e-12, rtol=0)
{
    'rank_H': H_RANK,
    'nullity': NULLSPACE_BASIS.shape[0],
    'kernel_states': len(KERNEL_STATES),
    'toy_validation': 'passed',
}


In [ ]:
mass_cache = {}
exact_rows = []
for row in tqdm(matched_cases.itertuples(index=False), total=len(matched_cases), desc='Exact logical masses', disable=CONFIG['print_level'] == 1):
    context = context_for(row.p)
    key = (float(row.p), row.syndrome)
    if key not in mass_cache:
        mass_cache[key] = compute_exact_logical_masses(context.H, context.L, bit_array(row.syndrome), context.priors)
    log_Z, probabilities, rank, nullity = mass_cache[key]
    true_sector_index = int(row.true_sector, 2)
    metrics = logical_ambiguity_metrics(log_Z, true_sector_index)
    exact_rows.append({
        'sample_id': row.sample_id,
        'matched_pair_id': row.matched_pair_id,
        'match_family': row.match_family,
        'outcome_class': row.outcome_class,
        'p': row.p,
        'error_weight': row.error_weight,
        'syndrome_weight': row.syndrome_weight,
        'residual_weight': row.residual_weight,
        'syndrome': row.syndrome,
        'true_sector': row.true_sector,
        'benchmark_sector': row.benchmark_sector,
        'max_normalized_late_residual': row.max_normalized_late_residual,
        'rank_H': rank,
        'nullity_H': nullity,
        **{f'log_Z_{sector}': log_Z[index] for index, sector in enumerate(SECTORS)},
        **{f'sector_probability_{sector}': probabilities[index] for index, sector in enumerate(SECTORS)},
        **metrics,
    })

exact_cases = pd.DataFrame(exact_rows)
assert exact_cases.sample_id.is_unique and exact_cases.groupby('matched_pair_id').size().eq(2).all()
assert np.allclose(exact_cases[[f'sector_probability_{sector}' for sector in SECTORS]].sum(axis=1), 1.0)

exact_cases.to_csv(_data_paths.output('exact_logical_mass_cases_csv'), index=False)
exact_mass_summary = pd.DataFrame([{
    'cases': len(exact_cases),
    'unique_p_syndrome_computations': len(mass_cache),
    'rank_H': H_RANK,
    'nullity_H': NULLSPACE_BASIS.shape[0],
    'configurations_per_syndrome': len(KERNEL_STATES),
    'maximum_probability_normalization_error': float(np.max(np.abs(exact_cases[[f'sector_probability_{sector}' for sector in SECTORS]].sum(axis=1) - 1))),
}])

exact_mass_summary.to_csv(_data_paths.output('exact_logical_mass_summary_csv'), index=False)
exact_mass_summary


## Experiment 1 analyses

Compare logical mass gap, entropy and BP residual on the same selected cases. Paired differences and predictive summaries test whether ambiguity distinguishes stable wrong endpoints better than message motion does.


In [ ]:
ambiguity_metrics = ['top_two_log_mass_gap', 'logical_entropy', 'normalized_logical_entropy', 'true_sector_gap']
group_rows = []
for outcome, group in exact_cases.groupby('outcome_class'):
    for metric in ambiguity_metrics:
        values = group[metric]
        group_rows.append({
            'outcome_class': outcome,
            'metric': metric,
            'count': len(values),
            'mean': values.mean(),
            'median': values.median(),
            'std': values.std(),
            'q25': values.quantile(0.25),
            'q75': values.quantile(0.75),
        })

logical_ambiguity_group_summary = pd.DataFrame(group_rows)
logical_ambiguity_group_summary.to_csv(_data_paths.output('logical_ambiguity_group_summary_csv'), index=False)

def bootstrap_paired_mean_interval(
    left,
    right,
    repetitions,
    seed,
    description,
):
    difference = np.asarray(left, dtype=float) - np.asarray(right, dtype=float)
    rng = np.random.default_rng(seed)
    estimates = np.empty(repetitions)

    for index in tqdm(range(repetitions), desc=description, leave=False, disable=CONFIG['print_level'] == 1):
        estimates[index] = rng.choice(difference, size=len(difference), replace=True).mean()
    return (difference.mean(), np.quantile(estimates, 0.025), np.quantile(estimates, 0.975))

stable_pivot = exact_cases[exact_cases.match_family == 'stable'].pivot(index='matched_pair_id', columns='outcome_class', values=ambiguity_metrics)
paired_rows = []
for metric in ambiguity_metrics:
    correct = stable_pivot[metric]['stable-valid-correct']
    wrong = stable_pivot[metric]['stable-valid-logical-wrong']
    mean_difference, ci_low, ci_high = bootstrap_paired_mean_interval(wrong, correct, CONFIG['bootstrap_repetitions'], CONFIG['random_seed'], f'Bootstrap ambiguity {metric}')
    statistic, p_value = wilcoxon(wrong, correct)
    paired_rows.append({
        'metric': metric,
        'difference_definition': 'stable_wrong_minus_stable_correct',
        'pairs': len(wrong),
        'mean_paired_difference': mean_difference,
        'bootstrap_ci_low': ci_low,
        'bootstrap_ci_high': ci_high,
        'wilcoxon_statistic': statistic,
        'wilcoxon_p_value': p_value,
    })

logical_ambiguity_paired = pd.DataFrame(paired_rows)
logical_ambiguity_paired.to_csv(_data_paths.output('logical_ambiguity_paired_comparison_csv'), index=False)
valid_exact = exact_cases.copy()
target = valid_exact.outcome_class.str.endswith('logical-wrong').astype(int).to_numpy()
features = {
    'late_residual': np.log10(np.maximum(valid_exact.max_normalized_late_residual.to_numpy(), np.finfo(float).tiny))[:, None],
    'negative_logical_gap': (-valid_exact.top_two_log_mass_gap.to_numpy())[:, None],
    'logical_entropy': valid_exact.normalized_logical_entropy.to_numpy()[:, None],
    'late_residual_plus_gap': np.column_stack([np.log10(np.maximum(valid_exact.max_normalized_late_residual, np.finfo(float).tiny)), valid_exact.top_two_log_mass_gap]),
    'late_residual_plus_entropy': np.column_stack([np.log10(np.maximum(valid_exact.max_normalized_late_residual, np.finfo(float).tiny)), valid_exact.normalized_logical_entropy]),
}

folds = StratifiedKFold(CONFIG['cv_folds'], shuffle=True, random_state=CONFIG['random_seed'])
auc_rows = []
class_indices = [np.flatnonzero(target == value) for value in (0, 1)]

for model_name, feature in features.items():
    if feature.shape[1] == 1:
        score = feature[:, 0]
    else:
        model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
        score = cross_val_predict(model, feature, target, cv=folds, method='predict_proba')[:, 1]

    auc = roc_auc_score(target, score)
    rng = np.random.default_rng(CONFIG['random_seed'] + len(auc_rows))
    bootstrap_auc = np.empty(CONFIG['bootstrap_repetitions'])

    for repetition in tqdm(range(CONFIG['bootstrap_repetitions']), desc=f'Bootstrap AUROC {model_name}', leave=False, disable=CONFIG['print_level'] == 1):
        selected = np.concatenate([rng.choice(indices, size=len(indices), replace=True) for indices in class_indices])
        bootstrap_auc[repetition] = roc_auc_score(target[selected], score[selected])

    auc_rows.append({
        'model': model_name,
        'cases': len(target),
        'auroc': auc,
        'bootstrap_ci_low': np.quantile(bootstrap_auc, 0.025),
        'bootstrap_ci_high': np.quantile(bootstrap_auc, 0.975),
        'cross_validated': feature.shape[1] > 1,
    })

logical_ambiguity_auc = pd.DataFrame(auc_rows)
logical_ambiguity_auc.to_csv(_data_paths.output('logical_ambiguity_auc_csv'), index=False)
conditional_rows = []

for metric, direction in [('top_two_log_mass_gap', 'below'), ('normalized_logical_entropy', 'above')]:
    for threshold in np.unique(np.quantile(valid_exact[metric], np.linspace(0.1, 0.9, 17))):
        selected = valid_exact[metric] < threshold if direction == 'below' else valid_exact[metric] > threshold
        subset = valid_exact[selected]
        conditional_rows.append({
            'metric': metric,
            'direction': direction,
            'threshold': threshold,
            'selected_cases': len(subset),
            'matched_case_control_logical_wrong_fraction': subset.outcome_class.str.endswith('logical-wrong').mean(),
        })

logical_ambiguity_conditional = pd.DataFrame(conditional_rows)
logical_ambiguity_conditional.to_csv(_data_paths.output('logical_ambiguity_conditional_curves_csv'), index=False)
logical_ambiguity_auc


## Experiment 2: logical susceptibility

Perturb selected variables and compare instability-targeted, random and reliability-based choices. Record syndrome consistency, sector changes and correct-sector discovery separately; an inconsistent output is not a successful logical switch.


In [ ]:
STRATEGIES = ['instability', 'low_reliability', 'stable', 'random', 'ambiguity_relevance']

def ordered_variables(strategy, instability, posterior_llr):
    variable_ids = np.arange(len(instability))
    if strategy == 'instability':
        score = -instability
    elif strategy == 'low_reliability':
        score = np.abs(posterior_llr)
    elif strategy == 'stable':
        score = instability
    elif strategy == 'ambiguity_relevance':
        score = -(instability / (np.abs(posterior_llr) + np.finfo(float).eps))
    else:
        raise AssertionError(strategy)

    return np.lexsort((variable_ids, score))

def perturbed_probabilities(
    prior_llr,
    selected,
    standard_noise,
    eta,
):
    perturbed = prior_llr.copy()
    sigma = eta * np.median(np.abs(prior_llr[selected]))
    perturbed[selected] += sigma * standard_noise
    probabilities = 1.0 / (1.0 + np.exp(perturbed))

    return np.clip(probabilities, CONFIG['probability_clip'], 1 - CONFIG['probability_clip'])

def normalized_sector_entropy(labels):
    counts = pd.Series(labels).value_counts().to_numpy(dtype=float)
    probabilities = counts / counts.sum()
    return float(-np.sum(probabilities * np.log(probabilities)) / np.log(4))


In [ ]:
susceptibility_input = exact_cases[exact_cases.match_family == 'stable'].copy()
assert len(susceptibility_input) == 2 * CONFIG['stable_matched_pairs']
trial_columns = ['sample_id', 'matched_pair_id', 'outcome_class', 'p', 'error_weight', 'syndrome_weight', 'selection_strategy', 'K', 'eta', 'trial', 'random_subset_id', 'baseline_logical_sector', 'perturbed_logical_sector', 'syndrome_consistent', 'logical_switched', 'correct_sector_discovered']
trial_path = _data_paths.output('logical_perturbation_trials_csv')
summary_rows = []
first_trial_batch = True

for sample in tqdm(susceptibility_input.itertuples(index=False), total=len(susceptibility_input), desc='Logical susceptibility', disable=CONFIG['print_level'] == 1):
    source = degeneracy_population.loc[degeneracy_population.sample_id == sample.sample_id].iloc[0]
    context = context_for(source.p)
    syndrome = bit_array(source.syndrome)
    truth = source.true_sector
    trace = trace_min_sum(context, syndrome, CONFIG['long_iterations'])
    instability = variable_instability_score(context, trace)
    posterior_llr = trace.llr[CONFIG['diagnostic_iterations']]
    decoder = BpDecoder(
        context.H,
        error_channel=context.priors,
        max_iter=CONFIG['benchmark_iterations'],
        bp_method='minimum_sum',
        ms_scaling_factor=CONFIG['ms_scaling_factor'],
        schedule='parallel',
        input_vector_type='syndrome',
    )
    baseline_hard = np.asarray(decoder.decode(syndrome), dtype=np.uint8)
    assert np.array_equal(context.H @ baseline_hard % 2, syndrome)
    baseline_sector = logical_label(context, baseline_hard)
    assert baseline_sector == source.benchmark_sector
    orders = {strategy: ordered_variables(strategy, instability, posterior_llr) for strategy in STRATEGIES if strategy != 'random'}
    sample_trial_rows = []

    for k_index, K in enumerate(CONFIG['susceptibility_k_values']):
        assert K <= context.H.shape[1]
        fixed_selected = {strategy: order[:K] for strategy, order in orders.items()}
        random_rng = np.random.default_rng(CONFIG['random_seed'] + int(sample.sample_id) * 1009 + K)
        random_subsets = [np.sort(random_rng.choice(context.H.shape[1], size=K, replace=False)) for _ in range(CONFIG['random_subset_replicates'])]

        for eta_index, eta in enumerate(CONFIG['perturbation_eta_values']):
            noise_rng = np.random.default_rng(CONFIG['random_seed'] + int(sample.sample_id) * 1000003 + k_index * 101 + eta_index)
            shared_noise = noise_rng.normal(size=(CONFIG['num_perturbations'], K))
            for strategy in STRATEGIES:
                labels = []
                valid_count = 0
                valid_switch_count = 0
                correct_discovery_count = 0
                selected_description = None

                for trial in range(CONFIG['num_perturbations']):
                    random_subset_id = trial % CONFIG['random_subset_replicates'] if strategy == 'random' else -1
                    selected = random_subsets[random_subset_id] if strategy == 'random' else fixed_selected[strategy]
                    selected_description = json.dumps(selected.astype(int).tolist()) if strategy != 'random' else 'multiple_random_subsets'
                    probabilities = perturbed_probabilities(context.prior_llr, selected, shared_noise[trial], eta)
                    assert np.all((probabilities > 0) & (probabilities < 1))
                    decoder.update_channel_probs(probabilities)
                    correction = np.asarray(decoder.decode(syndrome), dtype=np.uint8)
                    valid = np.array_equal(context.H @ correction % 2, syndrome)
                    label = logical_label(context, correction)
                    switched = label != baseline_sector
                    discovered = valid and label == truth
                    labels.append(label)
                    valid_count += int(valid)
                    valid_switch_count += int(valid and switched)
                    correct_discovery_count += int(discovered)
                    sample_trial_rows.append({
                        'sample_id': sample.sample_id,
                        'matched_pair_id': sample.matched_pair_id,
                        'outcome_class': sample.outcome_class,
                        'p': sample.p,
                        'error_weight': sample.error_weight,
                        'syndrome_weight': sample.syndrome_weight,
                        'selection_strategy': strategy,
                        'K': K,
                        'eta': eta,
                        'trial': trial,
                        'random_subset_id': random_subset_id,
                        'baseline_logical_sector': baseline_sector,
                        'perturbed_logical_sector': label,
                        'syndrome_consistent': int(valid),
                        'logical_switched': int(switched),
                        'correct_sector_discovered': int(discovered),
                    })

                summary_rows.append({
                    'sample_id': sample.sample_id,
                    'matched_pair_id': sample.matched_pair_id,
                    'outcome_class': sample.outcome_class,
                    'p': sample.p,
                    'error_weight': sample.error_weight,
                    'syndrome_weight': sample.syndrome_weight,
                    'selection_strategy': strategy,
                    'K': K,
                    'eta': eta,
                    'num_perturbations': CONFIG['num_perturbations'],
                    'selected_variables': selected_description,
                    'baseline_logical_sector': baseline_sector,
                    'baseline_syndrome_consistent': 1,
                    'logical_switch_fraction': np.mean(np.asarray(labels) != baseline_sector),
                    'valid_logical_switch_fraction': valid_switch_count / valid_count if valid_count else np.nan,
                    'valid_output_fraction': valid_count / CONFIG['num_perturbations'],
                    'perturbation_logical_entropy': normalized_sector_entropy(labels) * np.log(4),
                    'normalized_perturbation_logical_entropy': normalized_sector_entropy(labels),
                    'correct_sector_discovery_fraction': correct_discovery_count / CONFIG['num_perturbations'],
                    'num_unique_logical_sectors': len(set(labels)),
                })

    pd.DataFrame(sample_trial_rows, columns=trial_columns).to_csv(trial_path, mode='w' if first_trial_batch else 'a', header=first_trial_batch, index=False)
    first_trial_batch = False

susceptibility_cases = pd.DataFrame(summary_rows)
expected_rows = len(susceptibility_input) * len(STRATEGIES) * len(CONFIG['susceptibility_k_values']) * len(CONFIG['perturbation_eta_values'])
assert len(susceptibility_cases) == expected_rows
assert susceptibility_cases.groupby(['sample_id', 'selection_strategy', 'K', 'eta']).size().eq(1).all()
rate_columns = ['logical_switch_fraction', 'valid_logical_switch_fraction', 'valid_output_fraction', 'correct_sector_discovery_fraction']
assert susceptibility_cases[rate_columns].dropna().apply(lambda values: values.between(0, 1).all()).all()

susceptibility_cases.to_csv(_data_paths.output('logical_susceptibility_cases_csv'), index=False)
susceptibility_cases.head()


## Paired susceptibility and joint ambiguity analysis

Combine the matched susceptibility measurements with sector ambiguity. A relationship between them supports a diagnostic association, not a claim that perturbation repairs every ambiguous instance.


In [ ]:
def bootstrap_mean_interval(values, repetitions, seed):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    estimates = np.asarray([rng.choice(values, size=len(values), replace=True).mean() for _ in range(repetitions)])

    return np.quantile(estimates, [0.025, 0.975])

aggregate_rows = []
grouped = susceptibility_cases.groupby(['outcome_class', 'selection_strategy', 'K', 'eta'], sort=True)
for group_index, (keys, group) in enumerate(tqdm(grouped, total=grouped.ngroups, desc='Susceptibility confidence intervals', disable=CONFIG['print_level'] == 1)):
    for metric in ['valid_logical_switch_fraction', 'valid_output_fraction', 'correct_sector_discovery_fraction', 'normalized_perturbation_logical_entropy']:
        values = group[metric].dropna().to_numpy()
        ci_low, ci_high = bootstrap_mean_interval(values, CONFIG['bootstrap_repetitions'], CONFIG['random_seed'] + group_index)
        aggregate_rows.append({
            'outcome_class': keys[0],
            'selection_strategy': keys[1],
            'K': keys[2],
            'eta': keys[3],
            'metric': metric,
            'cases': len(values),
            'mean': values.mean(),
            'median': np.median(values),
            'bootstrap_ci_low': ci_low,
            'bootstrap_ci_high': ci_high,
        })

logical_susceptibility_summary = pd.DataFrame(aggregate_rows)
logical_susceptibility_summary.to_csv(_data_paths.output('logical_susceptibility_summary_csv'), index=False)
comparison_rows = []

for (outcome, K, eta), group in susceptibility_cases.groupby(['outcome_class', 'K', 'eta']):
    for metric in ['valid_logical_switch_fraction', 'correct_sector_discovery_fraction']:
        pivot = group.pivot(index='sample_id', columns='selection_strategy', values=metric)
        for control in ['random', 'low_reliability']:
            paired = pivot[['instability', control]].dropna()
            difference = paired.instability - paired[control]
            ci_low, ci_high = bootstrap_mean_interval(difference, CONFIG['bootstrap_repetitions'], CONFIG['random_seed'] + K)

            if np.allclose(difference, 0):
                statistic, p_value = (0.0, 1.0)
            else:
                statistic, p_value = wilcoxon(paired.instability, paired[control])
            comparison_rows.append({
                'outcome_class': outcome,
                'K': K,
                'eta': eta,
                'metric': metric,
                'comparison': f'instability_minus_{control}',
                'paired_cases': len(paired),
                'mean_paired_difference': difference.mean(),
                'median_paired_difference': difference.median(),
                'bootstrap_ci_low': ci_low,
                'bootstrap_ci_high': ci_high,
                'wilcoxon_statistic': statistic,
                'wilcoxon_p_value': p_value,
            })

logical_susceptibility_comparison = pd.DataFrame(comparison_rows)
logical_susceptibility_comparison.to_csv(_data_paths.output('logical_susceptibility_group_comparison_csv'), index=False)
joint = susceptibility_cases.merge(exact_cases[['sample_id', 'top_two_log_mass_gap', 'normalized_logical_entropy']], on='sample_id', how='left', validate='many_to_one')
correlation_rows = []

for (outcome, strategy, K, eta), group in joint.groupby(['outcome_class', 'selection_strategy', 'K', 'eta']):
    valid = group.dropna(subset=['valid_logical_switch_fraction'])
    for ambiguity_metric in ['top_two_log_mass_gap', 'normalized_logical_entropy']:
        correlation, p_value = spearmanr(valid.valid_logical_switch_fraction, valid[ambiguity_metric])
        correlation_rows.append({
            'outcome_class': outcome,
            'selection_strategy': strategy,
            'K': K,
            'eta': eta,
            'susceptibility_metric': 'valid_logical_switch_fraction',
            'ambiguity_metric': ambiguity_metric,
            'cases': len(valid),
            'spearman_correlation': correlation,
            'spearman_p_value': p_value,
        })

susceptibility_vs_gap = pd.DataFrame(correlation_rows)
susceptibility_vs_gap.to_csv(_data_paths.output('susceptibility_vs_logical_gap_csv'), index=False)
logical_susceptibility_comparison.head()


## Main figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
sns.scatterplot(
    data=exact_cases,
    x='max_normalized_late_residual',
    y='top_two_log_mass_gap',
    hue='outcome_class',
    ax=axes[0],
)
sns.scatterplot(
    data=exact_cases,
    x='max_normalized_late_residual',
    y='normalized_logical_entropy',
    hue='outcome_class',
    ax=axes[1],
)

for ax in axes:
    ax.set_xscale('log')
axes[0].set_title('Instability vs exact logical gap')
axes[1].set_title('Instability vs exact logical entropy')

plt.show()
stable_exact = exact_cases[exact_cases.match_family == 'stable']
fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
sns.violinplot(
    data=stable_exact,
    x='outcome_class',
    y='top_two_log_mass_gap',
    inner='box',
    ax=axes[0],
)
sns.violinplot(
    data=stable_exact,
    x='outcome_class',
    y='normalized_logical_entropy',
    inner='box',
    ax=axes[1],
)
sns.histplot(
    data=stable_exact,
    x='true_sector_gap',
    hue='outcome_class',
    element='step',
    stat='density',
    common_norm=False,
    ax=axes[2],
)

for ax in axes[:2]:
    ax.tick_params(axis='x', rotation=20)
plt.show()
preferred = logical_susceptibility_summary[(logical_susceptibility_summary.K == 8) & logical_susceptibility_summary.eta.isin([0.02, 0.05, 0.1]) & (logical_susceptibility_summary.metric == 'valid_logical_switch_fraction')]
preferred_bar = preferred[preferred.eta.eq(0.05)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
sns.barplot(
    data=preferred_bar,
    x='selection_strategy',
    y='mean',
    hue='outcome_class',
    ax=axes[0],
)
sns.lineplot(
    data=preferred,
    x='eta',
    y='mean',
    hue='selection_strategy',
    style='outcome_class',
    marker='o',
    ax=axes[1],
)
axes[0].tick_params(axis='x', rotation=25)
axes[0].set_title('Valid logical susceptibility by selection')
axes[1].set_title('Susceptibility vs perturbation strength')

plt.show()
wrong_discovery = logical_susceptibility_summary[logical_susceptibility_summary.outcome_class.eq('stable-valid-logical-wrong') & logical_susceptibility_summary.metric.eq('correct_sector_discovery_fraction')]
entropy_plot = logical_susceptibility_summary[logical_susceptibility_summary.metric.eq('normalized_perturbation_logical_entropy') & logical_susceptibility_summary.K.eq(8) & logical_susceptibility_summary.eta.eq(0.05)]
size_plot = logical_susceptibility_summary[logical_susceptibility_summary.metric.eq('valid_logical_switch_fraction') & logical_susceptibility_summary.eta.eq(0.05)]

fig, axes = plt.subplots(1, 3, figsize=(17, 5), constrained_layout=True)
sns.lineplot(
    data=size_plot,
    x='K',
    y='mean',
    hue='selection_strategy',
    style='outcome_class',
    marker='o',
    ax=axes[0],
)
sns.lineplot(
    data=wrong_discovery,
    x='eta',
    y='mean',
    hue='selection_strategy',
    style='K',
    marker='o',
    ax=axes[1],
)
sns.barplot(
    data=entropy_plot,
    x='selection_strategy',
    y='mean',
    hue='outcome_class',
    ax=axes[2],
)
axes[0].set_title('Susceptibility vs subset size')
axes[1].set_title('Correct-sector discovery')
axes[2].set_title('Perturbation logical entropy')
axes[2].tick_params(axis='x', rotation=25)

plt.show()
joint_preferred = joint[(joint.K == 8) & (joint.eta == 0.05)]
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
sns.scatterplot(
    data=joint_preferred,
    x='top_two_log_mass_gap',
    y='valid_logical_switch_fraction',
    hue='selection_strategy',
    style='outcome_class',
    ax=axes[0],
)
sns.scatterplot(
    data=joint_preferred,
    x='normalized_logical_entropy',
    y='valid_logical_switch_fraction',
    hue='selection_strategy',
    style='outcome_class',
    ax=axes[1],
)
axes[0].set_title('Susceptibility vs exact logical gap')
axes[1].set_title('Susceptibility vs exact logical entropy')

plt.show()


## Interpretation guardrails

The matched cohort is a case-control sample, so its class fraction is not a population logical-error rate. Exact masses use the BP Tanner-variable prior, not a different circuit-fault representation. A sampled true sector need not be the maximum-posterior sector. Correlation does not establish that degeneracy causes BP failure. Perturbation-induced syndrome-invalid outputs are excluded from valid logical susceptibility.